In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
from random import sample
import cv2


In [2]:
dat = pd.read_csv("../input/petfinder-pawpularity-score/train.csv")


In [3]:
y = dat["Pawpularity"].to_numpy()
y = y/100


In [4]:
### EXPLORE THE IMAGE SIZES

# r = []
# c = []
# for i in tqdm(dat["Id"]):
#     im = plt.imread("../input/petfinder-pawpularity-score/train/"+i+".jpg")
#     a,b = im.shape[:2]
#     r.append(a)
#     c.append(b)


In [5]:
# min(r),max(r),min(c),max(r)


In [6]:
class data:
    def __init__(self,path,x=128,y=128,labels=None):
        self.x = x
        self.y = y
        self.labels = labels
        self.image_list = [t.split(".")[0] for t in os.listdir(path)] 
        self.path = path
        self.batch = 0
        
    def load_batch(self,batch_size=1,shuffle=False):
        if shuffle:
            b = self.batch
            batch_list = self.image_list[b*batch_size:(b+1)*batch_size]
            self.batch = b+1
            if self.batch>len(self.image_list)//batch_size:
                self.batch=0
        else:
            batch_list = sample(self.image_list,batch_size)
        images = np.array([cv2.cvtColor(cv2.resize(cv2.imread(self.path+image+".jpg"),(128,128)),cv2.COLOR_BGR2RGB)/255 for image in batch_list])
        labels = self.labels.loc[batch_list].to_numpy()/100

        return images,labels

    
path = "../input/petfinder-pawpularity-score/train/"
labels = dat.set_index("Id")["Pawpularity"]
c = data(path,labels=labels)


In [7]:
from tensorflow import keras 
from tensorflow.keras import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.losses import *
from tensorflow.keras.optimizers import *


2026-01-07 06:03:50.461395: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767765830.474837      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767765830.479300      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-07 06:03:50.494921: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [8]:
model = Sequential()
model.add(Input(shape=(128,128,3)))
model.add(Conv2D(10,5))
model.add(MaxPooling2D())
model.add(Conv2D(20,3))
model.add(MaxPooling2D())
model.add(Flatten())
model.add(Dense(10, activation="relu"))
model.add(Dense(1,activation="relu"))


2026-01-07 06:03:57.199293: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1767765837.199760      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:85:00.0, compute capability: 8.6


In [9]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 124, 124, 10)   │           760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 62, 62, 10)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 60, 60, 20)     │         1,820 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 20)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 18000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │       180,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 182,601 (713.29 KB)

 Trainable params: 182,601 (713.29 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
model.compile(optimizer=Adam(1e-3),loss="mae")#,metrics="mae")


In [11]:
ids = dat["Id"].to_list()
train_ids = ids[:int(len(ids)*0.8)]
val_ids = ids[int(len(ids)*0.8):int(len(ids)*0.9)]
test_ids = ids[int(len(ids)*0.9):]


In [12]:
class data:
    def __init__(self,path,ids,x=128,y=128,labels=None):
        self.x = x
        self.y = y
        self.labels = labels
#         self.image_list = [t.split(".")[0] for t in os.listdir(path)] 
        self.image_list = ids
        self.path = path
        self.batch = 0
        
    def load_batch(self,batch_size=1,shuffle=False):
        if shuffle:
            b = self.batch
            batch_list = self.image_list[b*batch_size:(b+1)*batch_size]
            self.batch = b+1
            if self.batch>len(self.image_list)//batch_size:
                self.batch=0
        else:
            batch_list = sample(self.image_list,batch_size)
        images = np.array([cv2.cvtColor(cv2.resize(cv2.imread(self.path+image+".jpg"),(128,128)),cv2.COLOR_BGR2RGB)/255 for image in batch_list])
        labels = self.labels.loc[batch_list].to_numpy()/100

        return images,labels
    
    def loader(self,batch_size=1,shuffle=False):
        while True:
            x,y = self.load_batch(batch_size,shuffle)
            yield x,y
    
path = "../input/petfinder-pawpularity-score/train/"
labels = dat.set_index("Id")["Pawpularity"]
c = data(path,labels=labels,ids=ids)
c_train = data(path,labels=labels,ids=train_ids)
c_val = data(path,labels=labels,ids=val_ids)
c_test = data(path,labels=labels,ids=test_ids)


In [13]:
# model.fit_generator(c_train.loader(batch_size=128),validation_data=c_val.loader(batch_size=32),validation_steps=50,steps_per_epoch=100,epochs=2)


In [14]:
model.fit_generator(c.loader(batch_size=128),steps_per_epoch=100,epochs=2)
model.fit_generator(c.loader(batch_size=32),steps_per_epoch=100,epochs=1)


AttributeError: 'Sequential' object has no attribute 'fit_generator'

In [15]:
### VISUALIZE THE DISTRIBUTION OF ERROR (MAE)
# images = np.array([cv2.cvtColor(cv2.resize(cv2.imread(path+image+".jpg"),(128,128)),cv2.COLOR_BGR2RGB)/255 for image in ids])
# y = labels.loc[ids]
# y_pred = model.predict(images)
# plt.hist(np.abs(y.to_numpy().reshape((-1,1))/100-y_pred))


In [16]:
testpath = "../input/petfinder-pawpularity-score/test/"
ids = [t.split(".")[0] for t in os.listdir(testpath)]
# images = np.array([cv2.cvtColor(cv2.resize(cv2.imread(testpath+image+".jpg"),(128,128)),cv2.COLOR_BGR2RGB)/255 for image in ids])
# y_pred = model.predict(images)*100


In [17]:
batch_size=200
y_pred=np.zeros((len(ids),1))
for i in range(0,len(ids)//batch_size):
    images = np.array([cv2.cvtColor(cv2.resize(cv2.imread(testpath+image+".jpg"),(128,128)),cv2.COLOR_BGR2RGB)/255 for image in ids[batch_size*i:batch_size*(i+1)]])
    y_pred[i*batch_size:(i+1)*batch_size] = model.predict(images)*100


I0000 00:00:1767765838.753256      62 service.cc:148] XLA service 0x7fdfac247830 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767765838.753287      62 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-07 06:03:58.759726: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1767765838.771154      62 cuda_dnn.cc:529] Loaded cuDNN version 90300


7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step 

I0000 00:00:1767765839.095832      62 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


[ WARN:0@10.987] global loadsave.cpp:275 findDecoder imread_('../input/petfinder-pawpularity-score/test/test.jpg'): can't open/read file: check file path/integrity


error: OpenCV(4.12.0) /io/opencv/modules/imgproc/src/resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'resize'


In [18]:
### A TEST TO SEE IF WE CAN LOAD ALL THE DATA IN THE MEMORY
# testpath = "../input/petfinder-pawpularity-score/train/"
# ids = [t.split(".")[0] for t in os.listdir(testpath)]
# images = np.array([cv2.cvtColor(cv2.resize(cv2.imread(testpath+image+".jpg"),(128,128)),cv2.COLOR_BGR2RGB)/255 for image in ids])
# y_pred = model.predict(images)*100


In [19]:
round2 = lambda x,y=None:round(x+1e-15,y)
y_pred = [round2(t[0],2) for t in y_pred]


In [20]:
df = pd.DataFrame()
df["Id"] = ids
df["Pawpularity"] = y_pred


In [21]:
df.to_csv("submission.csv",index=False)


In [22]:
df


,Id,Pawpularity
0,a5c4de4c29e2097889f0d4cbd566625c,25.50
1,2e5cda2c4cf0530423e8181b7f8c67bc,59.09
2,98df091086f51b9ea2b26fcccefe11bb,17.75
3,db56a293c25086ba38f801f674c45689,9.66
4,7f3583a21d03cbc6b705f71f57a1a8d1,33.48
...,...,...
988,41684d34abf5fa83f4b0cb7567544b12,0.00
989,aa5d26353816ded3a8a6625aeeabe88b,0.00
990,0e94c5206de8777a57b63a4a52a751ef,0.00
991,aa97597cfb624189f811081b16bd909b,0.00
